# 01 · Data Quality Checks & Cleaning

**Goal:** before answering any business question, confirm the Superstore data can be trusted, fix what can't, and save a clean table for analysis and Power BI.

**Source:** [Sample Superstore, Kaggle](https://www.kaggle.com/datasets/naveenkumar20bps1137/sample-superstore) · 9,994 order lines · US retailer · 2014–2017

**Approach:** every check is written in SQL (DuckDB), so the same logic would run on a production warehouse.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

# Work out the project root whether the notebook runs from /notebooks or the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = (ROOT / "data" / "raw" / "superstore.csv").as_posix()
PROCESSED = ROOT / "data" / "processed"
IMAGES = ROOT / "images"
PROCESSED.mkdir(parents=True, exist_ok=True)
IMAGES.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the raw file

The two date columns are read as plain text on purpose. If the database guesses a date format, mistakes are hidden; reading them as text lets us inspect them first.

In [2]:
con.sql(f"""
    CREATE OR REPLACE TABLE orders_raw AS
    SELECT * FROM read_csv('{RAW}', header = true,
                           types = {{'Order Date': 'VARCHAR', 'Ship Date': 'VARCHAR'}})
""")
con.sql("SELECT COUNT(*) AS total_rows FROM orders_raw").df()

,total_rows
0,9994


In [3]:
con.sql("DESCRIBE orders_raw").df()

,column_name,column_type,null,key,default,extra
0,Row ID,BIGINT,YES,None,None,None
1,Order ID,VARCHAR,YES,None,None,None
2,Order Date,VARCHAR,YES,None,None,None
3,Ship Date,VARCHAR,YES,None,None,None
4,Ship Mode,VARCHAR,YES,None,None,None
5,Customer ID,VARCHAR,YES,None,None,None
6,Segment,VARCHAR,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None
8,City,VARCHAR,YES,None,None,None
9,State,VARCHAR,YES,None,None,None


## 2. Column profile: nulls, ranges, unique values

`SUMMARIZE` returns min, max, approximate unique counts and the null percentage for every column in one query.

In [4]:
con.sql("SUMMARIZE orders_raw").df()[["column_name", "column_type", "min", "max", "approx_unique", "null_percentage"]]

,column_name,column_type,min,max,approx_unique,null_percentage
0,Row ID,BIGINT,1,9994,9427,0.00
1,Order ID,VARCHAR,CA-2014-100006,US-2017-169551,5270,0.00
2,Order Date,VARCHAR,01-01-2017,9/30/2017,1221,0.00
3,Ship Date,VARCHAR,01-01-2015,9/30/2017,1376,0.00
4,Ship Mode,VARCHAR,First Class,Standard Class,4,0.00
5,Customer ID,VARCHAR,AA-10315,ZD-21925,780,0.00
6,Segment,VARCHAR,Consumer,Home Office,3,0.00
7,Country,VARCHAR,United States,United States,1,0.00
8,City,VARCHAR,Aberdeen,Yuma,547,0.00
9,State,VARCHAR,Alabama,Wyoming,50,0.00


**Note:** `approx_unique` is an *estimate* (DuckDB uses HyperLogLog for speed). It reports ~9,400 unique Row IDs, which could look like duplicate IDs. The exact check below shows the IDs are unique.

In [5]:
con.sql("""
    SELECT COUNT(*)                 AS total_rows,
           COUNT(DISTINCT "Row ID") AS exact_unique_row_ids,
           SUM(CASE WHEN "Sales" IS NULL OR "Profit" IS NULL THEN 1 ELSE 0 END) AS rows_missing_money
    FROM orders_raw
""").df()

,total_rows,exact_unique_row_ids,rows_missing_money
0,9994,9994,0.00


## 3. Issue found: two different date formats in one column

Some dates look like `4/15/2017`, others like `11-08-2016`. This usually happens when a file is opened and re-saved in Excel.

In [6]:
con.sql("""
    SELECT CASE WHEN "Order Date" LIKE '%/%' THEN 'M/D/YYYY (slash)'
                ELSE 'MM-DD-YYYY (dash)' END                       AS date_format,
           COUNT(*)                                                AS order_lines,
           MAX(CAST(split_part(replace("Order Date", '-', '/'), '/', 1) AS INT)) AS max_first_part,
           MAX(CAST(split_part(replace("Order Date", '-', '/'), '/', 2) AS INT)) AS max_second_part
    FROM orders_raw
    GROUP BY 1
""").df()

,date_format,order_lines,max_first_part,max_second_part
0,MM-DD-YYYY (dash),4042,12,12
1,M/D/YYYY (slash),5952,12,31


**Is it month-first or day-first?** In both formats, the first part never goes above 12, so the first part must be the **month**.

Then we check the decision. If the dates are parsed month-first, every shipment should leave a few days after it was ordered. The next query confirms that.

In [7]:
con.sql("""
    WITH parsed AS (
        SELECT strptime(replace("Order Date", '-', '/'), '%m/%d/%Y')::DATE AS order_date,
               strptime(replace("Ship Date",  '-', '/'), '%m/%d/%Y')::DATE AS ship_date
        FROM orders_raw
    )
    SELECT MIN(date_diff('day', order_date, ship_date)) AS min_days_to_ship,
           MAX(date_diff('day', order_date, ship_date)) AS max_days_to_ship,
           MIN(order_date) AS first_order,
           MAX(order_date) AS last_order
    FROM parsed
""").df()

,min_days_to_ship,max_days_to_ship,first_order,last_order
0,0,7,2014-01-03,2017-12-30


Shipping takes **0–7 days** for every order and the dates run Jan 2014 → Dec 2017. Both are realistic, so month-first parsing is correct.

## 4. Duplicate order lines

The same order line could appear twice with a different Row ID. We compare every column except Row ID.

In [8]:
con.sql("""
    SELECT "Order ID", "Product ID", "Sales", "Quantity", "Profit", COUNT(*) AS copies
    FROM orders_raw
    GROUP BY ALL
    HAVING COUNT(*) > 1
""").df()

,Order ID,Product ID,Sales,Quantity,Profit,copies
0,US-2014-150119,FUR-CH-10002965,281.37,2,-12.06,2


One line appears twice. Because every field matches, it is an **exact duplicate** and one copy will be removed.

## 5. Value sanity checks

In [9]:
con.sql("""
    SELECT MIN("Sales")    AS min_sales,     MAX("Sales")    AS max_sales,
           MIN("Quantity") AS min_qty,       MAX("Quantity") AS max_qty,
           MIN("Discount") AS min_discount,  MAX("Discount") AS max_discount,
           SUM(CASE WHEN "Profit" < 0 THEN 1 ELSE 0 END)       AS loss_making_lines,
           COUNT(DISTINCT "Country")                           AS countries
    FROM orders_raw
""").df()

,min_sales,max_sales,min_qty,max_qty,min_discount,max_discount,loss_making_lines,countries
0,0.44,"22,638.48",1,14,0.00,0.80,"1,871.00",1


- Sales are always positive and quantities are 1–14, so nothing looks impossible.
- Discounts run from 0% to **80%**. That is high but plausible, and it becomes the main thread of the analysis.
- Negative profit is **real** (loss-making sales), not an error, so it is kept.
- There is only one country, so the `Country` column adds nothing and is dropped.

## 6. Build the clean table

| Decision | Reason |
|---|---|
| Parse both date formats as month-first | Proven above (ship gap 0–7 days) |
| Remove 1 exact duplicate (`ROW_NUMBER`) | Every field identical, so the line would be double-counted |
| Rename to `snake_case` | Easier to query in SQL and Power BI |
| Add `ship_days`, `discount_band`, `is_loss`, `list_price_sales` | Reusable fields for analysis and the dashboard |
| Drop `Country` | Constant value |

In [10]:
con.sql("""
    CREATE OR REPLACE TABLE orders_clean AS
    WITH parsed AS (
        SELECT "Row ID"        AS row_id,
               "Order ID"      AS order_id,
               strptime(replace("Order Date", '-', '/'), '%m/%d/%Y')::DATE AS order_date,
               strptime(replace("Ship Date",  '-', '/'), '%m/%d/%Y')::DATE AS ship_date,
               "Ship Mode"     AS ship_mode,
               "Customer ID"   AS customer_id,
               "Segment"       AS segment,
               "City"          AS city,
               "State"         AS state,
               "Region"        AS region,
               "Product ID"    AS product_id,
               "Category"      AS category,
               "Sub-Category"  AS sub_category,
               "Product Name"  AS product_name,
               "Sales"         AS sales,
               "Quantity"      AS quantity,
               "Discount"      AS discount,
               "Profit"        AS profit
        FROM orders_raw
    ),
    deduped AS (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY order_id, order_date, ship_date, ship_mode, customer_id,
                                product_id, sales, quantity, discount, profit
                   ORDER BY row_id) AS copy_number
        FROM parsed
    )
    SELECT * EXCLUDE (copy_number),
           date_diff('day', order_date, ship_date)            AS ship_days,
           year(order_date)                                   AS order_year,
           CAST(date_trunc('month', order_date) AS DATE)      AS order_month,
           CASE WHEN discount = 0    THEN '0%'
                WHEN discount <= 0.2 THEN '1-20%'
                WHEN discount <= 0.3 THEN '21-30%'
                WHEN discount <= 0.5 THEN '31-50%'
                ELSE '51-80%' END                             AS discount_band,
           profit < 0                                         AS is_loss,
           ROUND(sales / (1 - discount), 4)                   AS list_price_sales
    FROM deduped
    WHERE copy_number = 1
""")
con.sql("SELECT COUNT(*) AS clean_rows FROM orders_clean").df()

,clean_rows
0,9993


## 7. Validate the result

In [11]:
con.sql("""
    SELECT COUNT(*)                      AS rows,
           COUNT(DISTINCT order_id)      AS orders,
           MIN(order_date)               AS first_order,
           MAX(order_date)               AS last_order,
           MIN(ship_days)                AS min_ship_days,
           MAX(ship_days)                AS max_ship_days,
           ROUND(SUM(sales), 2)          AS total_sales,
           ROUND(SUM(profit), 2)         AS total_profit
    FROM orders_clean
""").df()

,rows,orders,first_order,last_order,min_ship_days,max_ship_days,total_sales,total_profit
0,9993,5009,2014-01-03,2017-12-30,0,7,"2,296,919.49","286,409.08"


In [12]:
out = (PROCESSED / "superstore_clean.csv").as_posix()
con.sql(f"COPY orders_clean TO '{out}' (HEADER, DELIMITER ',')")
print("Saved:", out)

Saved: C:/Users/SUDHEER/Downloads/DataAnalytics/retail-profitability-analysis/data/processed/superstore_clean.csv


## Summary

- **9,994 → 9,993 rows**: one exact duplicate was removed.
- **Mixed date formats fixed**: 40% of dates were in a different format. Month-first parsing was proven by the 0–7 day ship gap.
- **No missing values** in any column.
- The clean table is saved to `data/processed/superstore_clean.csv` and feeds notebook 02 and the Power BI dashboard.